<a href="https://colab.research.google.com/github/millerfabiancanon-a11y/Business_Analytics/blob/main/ETL_Business_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗂️ Proyecto ETL – Business Analytics

## Análisis de Proyectos de Financiamiento del Banco Mundial

**Miller Fabian Cañon Cañon**


---
## 📌 Introducción

Un proceso **ETL (Extract, Transform, Load)** es el conjunto de pasos con los que se extraen datos de una fuente, se transforman (limpian, corrigen y estructuran) y finalmente se cargan en un formato listo para el análisis. Es la base de cualquier proyecto de Business Analytics, porque los datos casi nunca llegan listos para usarse: vienen con errores, formatos inconsistentes, información faltante o texto donde debería haber números.

Este notebook desarrolla las tres fases del proceso sobre un dataset real:

1. **Extract** – lectura de los datos originales desde un archivo CSV.
2. **Transform** – limpieza, corrección de tipos de datos, manejo de valores nulos, creación de una columna calculada y filtrado por reglas de negocio.
3. **Load** – guardado del resultado final en dos formatos aptos para análisis: Parquet y una base de datos SQLite.

### Selección de la fuente de datos

Para este proyecto se evaluaron distintos conjuntos de datos disponibles en Kaggle, priorizando aquellos que representaran un reto realista para la aplicación de un proceso ETL completo. Se seleccionó el dataset *"World Bank Projects"* porque, a diferencia de otros conjuntos más limpios o ya preprocesados, presentaba problemas típicos de datos empresariales del mundo real: columnas completamente vacías, valores sin registrar (nulos), inconsistencias en los nombres de las columnas (espacios, mayúsculas) y formatos de datos incorrectos (montos monetarios almacenados como texto, fechas sin tipificar). Esto permitió poner en práctica de forma integral las tres fases del proceso ETL, abordando escenarios reales de limpieza, validación y estructuración de datos.


---
## 📚 Importación de librerías

Antes de empezar, traemos las herramientas que vamos a usar en todo el proceso:

- **pandas**: para leer, limpiar y transformar los datos en forma de tabla.
- **sqlite3**: para guardar el resultado final en una base de datos, sin necesidad de instalar un servidor.
- **shutil**: para copiar los archivos finales a Google Drive y que no se pierdan al cerrar la sesión.


In [ ]:
import pandas as pd
import sqlite3
import shutil


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 📂 Fase 1 – Extract: carga y exploración inicial del dataset

En esta sección se conecta el notebook con Google Drive, se lee el archivo original (`projects_data.csv`) y se hace una primera revisión: número de filas y columnas, nombres de columnas y tipos de datos. Esta exploración inicial es la que permitió detectar, más adelante, todos los problemas que se corrigen en la fase de Transformación.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ruta = "/content/drive/MyDrive/projects_data.csv"


Se construye una función para leer el archivo, en lugar de repetir el código cada vez que se necesite. Además, incluye manejo de errores: si el archivo no se encuentra en la ruta indicada, el programa avisa con un mensaje claro en vez de detenerse de forma abrupta.


In [ ]:
def extraer_datos(ruta_archivo):
    """Lee un archivo CSV y devuelve un DataFrame de pandas."""
    try:
        df = pd.read_csv(ruta_archivo, low_memory=False)
        print("Datos leídos correctamente:", df.shape)
        return df
    except FileNotFoundError:
        print("No encontré el archivo en esta ruta:", ruta_archivo)
        return None


df = extraer_datos(ruta)


**Exploración inicial:** revisamos las dimensiones, los nombres de columnas y el tipo de dato de cada una. Esto es lo que nos permitió identificar, por ejemplo, que las columnas de montos de dinero estaban guardadas como texto en lugar de números.


In [ ]:
print(df.shape)
print(df.columns)
df.info()


---
## 🧹 Fase 2 – Transform: limpieza, corrección y estructuración

Esta es la fase más extensa del proceso. Se aplican, en orden, todas las correcciones necesarias para que el dataset quede consistente y listo para análisis.


### 2.1 Eliminación de columnas totalmente vacías

Se identificaron 14 columnas sin un solo dato registrado en las 18,248 filas (por ejemplo, `mjtheme1name` a `mjtheme5name`). No aportan ninguna información, así que se eliminan.


In [ ]:
columnas_vacias = df.columns[df.isnull().all()]
print("Columnas vacías encontradas:", columnas_vacias.tolist())

df = df.drop(columns=columnas_vacias)
print("Nuevas dimensiones:", df.shape)


### 2.2 Limpieza de nombres de columnas

Algunos nombres de columna traían espacios de más al final (por ejemplo, `'theme '`), lo cual puede causar errores al momento de referenciarlas en el código.


In [ ]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())


### 2.3 Eliminación de columna redundante

`status` y `projectstatusdisplay` contienen prácticamente la misma información (99.98% idénticas). Se conserva `status` por ser más clara y se elimina la otra, para no duplicar información.


In [ ]:
df = df.drop(columns=["projectstatusdisplay"])
print(df.shape)


### 2.4 Corrección de la columna `countryname`

El país venía duplicado dentro del mismo texto, separado por `;` (por ejemplo, `"World;World"`). Se conserva solo la primera aparición.


In [ ]:
df["countryname"] = df["countryname"].str.split(";").str[0]
print(df["countryname"].head())


### 2.5 Conversión de columnas monetarias a formato numérico

Las columnas de montos (`lendprojectcost`, `ibrdcommamt`, `idacommamt`, `totalamt`, `grantamt`) estaban guardadas como texto, con comas como separador de miles (por ejemplo, `"200,000,000"`). Se limpian y se convierten a número, ya que en ese formato no se pueden realizar cálculos.


In [ ]:
def limpiar_columna_dinero(columna):
    """Quita las comas de separación de miles y convierte el texto a número decimal."""
    columna = columna.str.replace(",", "")
    columna = columna.astype(float)
    return columna


columnas_dinero = ["totalamt", "lendprojectcost", "ibrdcommamt", "idacommamt", "grantamt"]
for columna in columnas_dinero:
    df[columna] = limpiar_columna_dinero(df[columna])

print(df[columnas_dinero].dtypes)


### 2.6 Conversión de columnas de fecha

`boardapprovaldate` y `closingdate` venían como texto (por ejemplo, `"2018-06-28T00:00:00Z"`). Se convierten a un tipo de dato de fecha real, lo que permite calcular duraciones y filtrar por rangos de tiempo.


In [ ]:
df["boardapprovaldate"] = pd.to_datetime(df["boardapprovaldate"])
df["closingdate"] = pd.to_datetime(df["closingdate"])

print(df[["boardapprovaldate", "closingdate"]].dtypes)


### 2.7 Eliminación de columnas con exceso de valores nulos

Se estableció un umbral de negocio: toda columna con más del **70% de valores nulos** se considera poco útil para el análisis y se elimina (por ejemplo, columnas de geolocalización específica que casi nunca estaban registradas).


In [ ]:
umbral_nulos = 70  # % de nulos a partir del cual se elimina una columna

porcentaje_nulos = df.isnull().mean() * 100
columnas_a_eliminar = porcentaje_nulos[porcentaje_nulos > umbral_nulos].index
print("Columnas eliminadas por exceso de nulos:", columnas_a_eliminar.tolist())

df = df.drop(columns=columnas_a_eliminar)
print("Nuevas dimensiones:", df.shape)


### 2.8 Tratamiento de valores nulos restantes

Para las columnas de texto que aún tenían nulos (por ejemplo, `sector2`, `financier`, `borrower`), se rellenan con la etiqueta `"Not specified"`, ya que el vacío representa información que simplemente no fue registrada. Para la columna numérica `lendprojectcost`, se rellena con la **mediana**, una medida robusta frente a valores extremos. Las fechas (`boardapprovaldate`, `closingdate`) se dejan sin rellenar de forma intencional: un proyecto sin fecha de cierre es información real (aún no ha cerrado), no un error.


In [ ]:
# Columnas de texto: se rellenan con una etiqueta explícita
columnas_texto = df.select_dtypes(include="object").columns
df[columnas_texto] = df[columnas_texto].fillna("Not specified")

# Columna numérica: se rellena con la mediana
mediana_costo = df["lendprojectcost"].median()
df["lendprojectcost"] = df["lendprojectcost"].fillna(mediana_costo)

print(df.isnull().sum())


### 2.9 Columna calculada: duración del proyecto

Se crea una nueva columna, `project_duration_days`, que calcula cuántos días transcurren entre la aprobación y el cierre de cada proyecto. Al revisar los resultados, se encontraron 14 proyectos con duración negativa (la fecha de cierre era anterior a la de aprobación), lo cual es una inconsistencia de los datos originales. En lugar de eliminar esas filas completas (y perder toda su información), se decidió dejar únicamente la duración de esos 14 casos como valor nulo, conservando el resto de los datos del proyecto intactos.


In [ ]:
df["project_duration_days"] = (df["closingdate"] - df["boardapprovaldate"]).dt.days

# Corrección de inconsistencias: duración negativa no tiene sentido de negocio
df.loc[df["project_duration_days"] < 0, "project_duration_days"] = None

print("Duración negativa restante:", (df["project_duration_days"] < 0).sum())
print(df[["boardapprovaldate", "closingdate", "project_duration_days"]].head())


### 2.10 Filtrado por regla de negocio

Se evaluó eliminar los proyectos con estado `"Closed"` (finalizados) y `totalamt = 0`. Antes de aplicar el filtro, se verificó que no se estuviera perdiendo información válida: de 2,479 proyectos en esa condición, 2,280 sí contaban con financiamiento real a través de donaciones (`grantamt > 0`), por lo que no debían eliminarse. El filtro final se ajustó para eliminar únicamente los proyectos cerrados que no tienen financiamiento registrado por **ningún** concepto (ni `totalamt` ni `grantamt`).


In [ ]:
condicion_eliminar = (df["status"] == "Closed") & (df["totalamt"] == 0) & (df["grantamt"] == 0)
print("Filas a eliminar:", condicion_eliminar.sum())

df = df[~condicion_eliminar]
print("Dimensiones finales del dataset limpio:", df.shape)


---
## 💾 Fase 3 – Load: carga de los datos limpios

Con el dataset ya limpio y estructurado, se guarda en dos formatos aptos para análisis posterior:

- **Parquet**: formato comprimido y eficiente, ideal para cargar rápidamente el dataset en herramientas de análisis.
- **SQLite**: una base de datos real, que permite hacer consultas tipo SQL sobre los datos sin necesidad de un servidor externo.

Finalmente, ambos archivos se copian a Google Drive para que no se pierdan al cerrar la sesión de Colab.


In [ ]:
# Guardar en formato Parquet
df.to_parquet("projects_clean.parquet", index=False)
print("Archivo Parquet guardado correctamente")

# Guardar en base de datos SQLite
conexion = sqlite3.connect("projects.db")
df.to_sql("wb_projects", conexion, if_exists="replace", index=False)
conexion.close()
print("Datos guardados en la base de datos SQLite")


In [ ]:
# Copiar ambos archivos a Google Drive para conservarlos
shutil.copy("projects_clean.parquet", "/content/drive/MyDrive/projects_clean.parquet")
shutil.copy("projects.db", "/content/drive/MyDrive/projects.db")
print("Archivos copiados a Drive")


**Verificación final:** se vuelve a leer el archivo Parquet guardado, para confirmar que el proceso completo (Extract → Transform → Load) se ejecutó correctamente y que el archivo final contiene la cantidad esperada de filas y columnas.


In [ ]:
prueba = pd.read_parquet("projects_clean.parquet")
print("Dimensiones del archivo final verificado:", prueba.shape)


---
## ✅ Conclusiones

El proceso ETL desarrollado permitió transformar un conjunto de datos crudo de 18,248 filas y 57 columnas —con columnas vacías, valores nulos, formatos de texto en lugar de números y fechas, e inconsistencias lógicas— en un dataset limpio, coherente y estructurado de 18,049 filas y 33 columnas, listo para ser utilizado en análisis posteriores.

Más allá de la ejecución técnica, cada decisión de limpieza (qué columnas eliminar, cómo tratar los valores nulos, qué hacer con las inconsistencias de fecha, y qué regla de filtrado aplicar) fue validada antes de aplicarse, evitando la pérdida de información válida — como ocurrió con los proyectos financiados por donación que inicialmente parecían no tener financiamiento. Este proceso refleja la importancia del ETL como etapa fundamental dentro del Business Analytics: sin datos confiables y bien estructurados, cualquier análisis o modelo posterior parte de una base débil.
